In [ ]:
# Ячейка 1 — встановлення та підключення Google Drive
!pip install requests beautifulsoup4 tqdm -q

import requests
from bs4 import BeautifulSoup
import os
import time
import re
import json
from datetime import datetime, timedelta
from urllib.parse import urlencode
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive підключено")

In [ ]:
# Ячейка 2 — налаштування

YEAR = 2025  # змінюй для кожного року

DOWNLOAD_FOLDER = "/content/drive/MyDrive/nrat_pdfs"
CHECKPOINT_FILE = os.path.join(DOWNLOAD_FOLDER, f"_progress_{YEAR}.json")

BASE_SEARCH_URL = "https://nrat.ukrintei.ua/searchdb/?"
BASE_PARAMS = {
    '_token': '6C0elymE1XyE8AOayLEsuYco3JewHUAF0DazKx9Y',
    'typeSearch2': 'ok',
    'typeCategory[]': '0',
    'lcSource': '',
    'authorSearch': '',
    'specialnistSearch[]': '0',
    'temaSearch2': '',
    'textSearch': '',
    'registrationNumberSearch': '',
    'firm_id': '0',
    'sortOrder': 'registration_date',
    'sortDir': 'desc',
    'tab': 'big'
}

DAYS_PER_CHUNK = 1

# НЕ ЗМЕНШУВАТИ — інакше бан на 1-2 дні
DELAY_BETWEEN_PAGES = 3
DELAY_BETWEEN_FILES = 1
DELAY_BETWEEN_DAYS  = 5

os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)

def day_folder_for(year, date_from):
    return os.path.join(DOWNLOAD_FOLDER, str(year), date_from[:7], date_from)

session = requests.Session()
session.headers.update({
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9,uk;q=0.8',
})

print(f"Рік: {YEAR}")
print("Папка:", os.path.join(DOWNLOAD_FOLDER, str(YEAR)))

In [ ]:
# Ячейка 3 — функції

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        try:
            with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
                return set(json.load(f).get('completed_dates', []))
        except Exception:
            return set()
    return set()

def save_checkpoint(completed_set):
    try:
        with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
            json.dump({'completed_dates': sorted(completed_set)}, f, ensure_ascii=False, indent=2)
    except Exception as e:
        print(f"  ⚠️ Не вдалося записати чекпоінт: {e}")

def mark_date_done(completed_set, date_str):
    completed_set.add(date_str)
    save_checkpoint(completed_set)

def refresh_token():
    try:
        r = session.get("https://nrat.ukrintei.ua/searchdb/", timeout=30)
        soup = BeautifulSoup(r.content, 'html.parser')
        tok = soup.find('input', attrs={'name': '_token'})
        if tok and tok.get('value'):
            BASE_PARAMS['_token'] = tok['value']
            print("  🔑 _token оновлено")
            return True
    except Exception as e:
        print(f"  ⚠️ Не вдалося оновити _token ({e})")
    return False

def generate_date_ranges(start_date_str, end_date_str, days_per_chunk):
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    ranges = []
    cur = start_date
    while cur <= end_date:
        cur_end = min(cur + timedelta(days=days_per_chunk - 1), end_date)
        ranges.append((cur.strftime("%Y-%m-%d"), cur_end.strftime("%Y-%m-%d")))
        cur = cur_end + timedelta(days=1)
    return ranges

def build_search_url(date_from, date_to, page=1):
    params = BASE_PARAMS.copy()
    params['dateFromSearch'] = date_from
    params['dateToSearch'] = date_to
    params['pa'] = str(page)  # параметр пагінації на сайті — 'pa', не 'page'
    return BASE_SEARCH_URL + urlencode(params, doseq=True)

def extract_total_results(soup):
    try:
        page_info = soup.find('div', class_='page_info')
        if page_info:
            m = re.search(r'Знайдено документів:\s*(\d+)', page_info.get_text(strip=True))
            if m:
                return int(m.group(1))
        m = re.search(r'Знайдено документів:\s*(\d+)', soup.get_text(" ", strip=True))
        if m:
            return int(m.group(1))
    except Exception:
        pass
    return 0

def get_search_results_page(url, retry_count=3):
    for attempt in range(retry_count):
        try:
            response = session.get(url, timeout=60)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')
            results = []
            for card in soup.find_all('div', class_='my-card-body'):
                link_tag = card.find('a', target='_blank')
                if link_tag and link_tag.get('href'):
                    reg_number = link_tag.get_text(strip=True)
                    detail_url = link_tag['href']
                    card_text = card.get_text(separator=' ', strip=True)
                    parts = card_text.split('Керівник:')
                    title = parts[0].strip() if len(parts) > 1 else card_text[:100].strip()
                    doc_id = detail_url.rstrip('/').split('/')[-1]
                    pdf_url = f"https://dir.ukrintei.ua/view/ok/{doc_id}"
                    results.append({
                        'registration': reg_number, 'detail_url': detail_url,
                        'pdf_url': pdf_url, 'title': title, 'doc_id': doc_id
                    })
            return results, soup
        except Exception as e:
            print(f"    ⚠️ спроба {attempt + 1}/{retry_count}: {e}")
            if attempt < retry_count - 1:
                time.sleep(5)
    return None, None

def download_pdf(pdf_url, registration, title, doc_id, folder, retry_count=2):
    safe_reg = re.sub(r'[^\w\-_.]', '_', registration)
    filename = re.sub(r'_+', '_', f"{safe_reg}_{doc_id}.pdf")
    filepath = os.path.join(folder, filename)

    if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
        return 'skip', filepath

    for attempt in range(retry_count):
        try:
            response = session.get(pdf_url, stream=True, timeout=60)
            if response.status_code == 404:
                return 'notpdf', None
            response.raise_for_status()
            with open(filepath, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
            if os.path.getsize(filepath) > 0:
                with open(filepath, 'rb') as f:
                    if f.read(4).startswith(b'%PDF'):
                        return 'ok', filepath
                os.remove(filepath)
                return 'notpdf', None
            else:
                os.remove(filepath)
                return 'empty', None
        except Exception as e:
            print(f"    ❌ Помилка завантаження: {e}")
            if attempt < retry_count - 1:
                time.sleep(3)
    return 'fail', None

print("✅ Функції завантажено")

In [ ]:
# Ячейка 4 — логіка

def scrape_day(date_from, date_to, day_folder):
    os.makedirs(day_folder, exist_ok=True)
    page = 1
    page_size = None
    total_pages = None
    seen_ids = set()
    stats = {'ok': 0, 'skip': 0, 'fail': 0, 'notpdf': 0}
    day_ok = True

    while True:
        url = build_search_url(date_from, date_to, page)
        results, soup = get_search_results_page(url)

        if results is None:
            print("    ⚠️ сторінка не завантажилась — день буде повторено")
            day_ok = False
            break

        if page == 1:
            total_results = extract_total_results(soup)
            page_size = len(results) if results else 10
            total_pages = ((total_results + 9) // 10) if total_results else None
            if total_results:
                print(f"    знайдено {total_results} рез., {total_pages} стор.")
            if total_results >= 1000:
                print("    ⚠️ 1000+ результатів — можливо, не всі потраплять у видачу")

        if not results:
            if page == 1:
                print("    немає результатів")
            break

        new_results = [r for r in results if r['doc_id'] not in seen_ids]
        if page > 1 and not new_results:
            break
        for r in new_results:
            seen_ids.add(r['doc_id'])

        for i, r in enumerate(new_results, 1):
            status, _ = download_pdf(r['pdf_url'], r['registration'], r['title'], r['doc_id'], day_folder)
            stats[status if status in stats else 'fail'] += 1
            if i < len(new_results):
                time.sleep(DELAY_BETWEEN_FILES)

        if total_pages is not None:
            go_next = page < total_pages
        else:
            go_next = bool(page_size and len(results) >= page_size)
        if not go_next or page >= 85:
            break

        page += 1
        time.sleep(DELAY_BETWEEN_PAGES)

    return stats, day_ok


def run_year(year):
    completed = load_checkpoint()
    print(f"📌 Вже завершено днів: {len(completed)}")
    refresh_token()

    date_ranges = generate_date_ranges(f"{year}-01-01", f"{year}-12-31", DAYS_PER_CHUNK)
    grand = {'ok': 0, 'skip': 0, 'fail': 0, 'notpdf': 0}
    incomplete = []

    print("\n" + "=" * 70)
    print(f"РІК {year} — {len(date_ranges)} днів")
    print("=" * 70)

    for idx, (date_from, date_to) in enumerate(date_ranges, 1):
        if date_from in completed:
            continue

        print(f"\n📅 {date_from}  ({idx}/{len(date_ranges)})")
        stats, day_ok = scrape_day(date_from, date_to, day_folder_for(year, date_from))
        for k in grand:
            grand[k] += stats[k]
        print(f"    підсумок дня: ✅{stats['ok']} ⏭{stats['skip']} 📄✗{stats['notpdf']} ❌{stats['fail']}"
              f"   |   ВСЬОГО за рік ✅{grand['ok']}")

        if day_ok:
            mark_date_done(completed, date_from)
        else:
            incomplete.append(date_from)
        time.sleep(DELAY_BETWEEN_DAYS)

    print("\n" + "=" * 70)
    print(f"РІК {year} ГОТОВО!")
    print(f"Завантажено нових PDF: {grand['ok']} | пропущено (вже було): {grand['skip']} | "
          f"без файлу: {grand['notpdf']} | помилок: {grand['fail']}")
    if incomplete:
        print(f"⚠️ Дні з помилкою мережі (запусти цю ячейку ще раз — доберуться): {len(incomplete)}")
        print("   ", incomplete)
    print("=" * 70)
    return grand

print("✅ Логіку завантажено. Запускай ячейку 'ЗІБРАТИ ВЕСЬ РІК'.")

In [ ]:
# ЗІБРАТИ ВЕСЬ РІК  (⚠️ виконується ГОДИНАМИ)
# Якщо Colab перервався — запусти ячейки 1–4 і цю знову: продовжить з місця зупинки.
year_stats = run_year(YEAR)